# Sentiment Analysis — LSTM

Recurrent neural network trained on the **IMDB movie reviews** corpus to classify text as **positive** or **negative**. Model + word index saved to `models/` for the website Demo.

## Part 0 — Setup

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import json, pickle
from pathlib import Path

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence

print('TensorFlow', tf.__version__)

## Part 1 — Load IMDB Reviews

Keeps the top 20 000 most frequent words. Reviews are already tokenised as integer sequences.

In [ ]:
VOCAB_SIZE = 20000
MAXLEN = 200

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
print('Train:', X_train.shape, '  Test:', X_test.shape)

### 1.1 Pad sequences to fixed length

In [ ]:
X_train = sequence.pad_sequences(X_train, maxlen=MAXLEN)
X_test  = sequence.pad_sequences(X_test,  maxlen=MAXLEN)
X_train.shape

### 1.2 Decode a sample (sanity check)

In [ ]:
word_index = imdb.get_word_index()
index_to_word = {v + 3: k for k, v in word_index.items()}
index_to_word[0] = '<pad>'
index_to_word[1] = '<start>'
index_to_word[2] = '<oov>'
decoded = ' '.join(index_to_word.get(i, '?') for i in X_train[0] if i != 0)
print(decoded[:300], '...')
print('Sentiment:', 'positive' if y_train[0] == 1 else 'negative')

## Part 2 — Build the LSTM

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(MAXLEN,)),
    tf.keras.layers.Embedding(VOCAB_SIZE, 64),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## Part 3 — Train

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=3,
    batch_size=128,
    validation_data=(X_test, y_test),
    verbose=1,
)

### 3.1 Plot training history

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['loss'], label='train')
ax[0].plot(history.history['val_loss'], label='val')
ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(history.history['accuracy'], label='train')
ax[1].plot(history.history['val_accuracy'], label='val')
ax[1].set_title('Accuracy'); ax[1].legend()
plt.tight_layout(); plt.show()

## Part 4 — Evaluate

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {acc:.4f}')

## Part 5 — Persist Model + Vocabulary

In [ ]:
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODELS_DIR / 'sentiment_lstm.keras'
VOCAB_PATH = MODELS_DIR / 'sentiment_vocab.json'

model.save(MODEL_PATH)
with open(VOCAB_PATH, 'w', encoding='utf-8') as f:
    json.dump({
        'word_index': word_index,
        'vocab_size': VOCAB_SIZE,
        'maxlen': MAXLEN,
    }, f)
print('Saved ->', MODEL_PATH, VOCAB_PATH)

## Part 6 — Predict Helper

In [ ]:
import re

def encode(text: str):
    tokens = re.findall(r'[a-z\']+', text.lower())
    ids = [word_index.get(t, 2) + 3 for t in tokens]
    return sequence.pad_sequences([ids], maxlen=MAXLEN)

def predict_sentiment(text: str):
    p = float(model.predict(encode(text), verbose=0)[0][0])
    return ('positive' if p > 0.5 else 'negative'), p

for sample in [
    'Absolutely loved this movie, the performances were stunning!',
    'Worst film I have seen in years. Boring and cliché.',
    'It was fine. Some scenes worked, some did not.',
]:
    label, p = predict_sentiment(sample)
    print(f'{p:.3f} -> {label.upper():8s} | {sample}')